# 🔀 Dynamic Middleware in LangChain
## `wrap_model_call`, `wrap_tool_call`, and `dynamic_prompt`

---

## 📖 What is this notebook?

This is a **reference tutorial** for the non-hook-style middleware in LangChain — the ones that **wrap** the actual LLM call or tool execution, or **dynamically generate** prompts. These are fundamentally different from node-style hooks (`@before_agent`, `@after_agent`).

---

## 🧠 The Three Middleware Families

| Family | Decorators | What they intercept | Analogy |
|---|---|---|---|
| 🪝 Node-style | `@before_agent`, `@after_agent` | State at node boundaries | Express `app.use()` — touch one side |
| 🔀 **Wrap-style** | `@wrap_model_call`, `@wrap_tool_call` | The actual invocation | Python decorator — wrap the whole call |
| ⚙️ **Config-style** | `@dynamic_prompt` | Prompt generation | Template engine — generate on the fly |

### 🔑 Key difference

- **Node-style** = "transform the state before/after a node runs"
- **Wrap-style** = "wrap the entire call — you see the request going in AND the response coming out, in one function"
- **Config-style** = "dynamically generate configuration (prompts, settings) based on runtime state"

---

## 🔀 Part 1: `wrap_model_call`

### What it does

Intercepts the **LLM invocation itself**. You receive a `ModelRequest` (containing messages, tools, model config) and a `handler` callback. You can:

- 🔄 **Swap the model** — route to different LLMs based on conversation state
- 🔧 **Modify parameters** — change temperature, max_tokens, etc.
- 🛡️ **Filter tools** — restrict which tools the LLM can see
- ⏱️ **Add timing/logging** — measure how long the LLM call takes
- 🔁 **Implement retries** — catch errors and retry with a different model

### The pattern

```python
@wrap_model_call
def my_wrapper(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    # 1. Inspect request (messages, tools, runtime, etc.)
    # 2. Optionally modify: request = request.override(model=..., tools=...)
    # 3. Call handler to proceed: response = handler(request)
    # 4. Optionally inspect/modify response
    return response
```

### `ModelRequest` — what you can read

| Field | Type | Description |
|---|---|---|
| `request.messages` | `list[BaseMessage]` | Shortcut for `state["messages"]` |
| `request.runtime` | `Runtime` | Access to `.context`, `.state`, `.config` |
| `request.tools` | `list[Tool]` | Tools currently available to the LLM |
| `request.model` | `BaseChatModel` | The current LLM instance |

### `request.override()` — what you can change

| Override | What it does |
|---|---|
| `request.override(model=new_model)` | Swap to a different LLM |
| `request.override(tools=filtered_tools)` | Change which tools the LLM sees |
| `request.override(messages=modified_msgs)` | Modify the messages sent to the LLM |

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### 🔄 Example 1A: Dynamic Model Selection

Route to a larger model when the conversation gets long.

**Why?** Short conversations are fine with a cheap, fast model. But as context grows, you need a model with a bigger context window and stronger reasoning to handle the accumulated history.

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def route_by_length(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Switch to a larger model when conversation exceeds 10 messages."""
    message_count = len(request.messages)

    if message_count > 10:
        request = request.override(model=large_model)
    else:
        request = request.override(model=standard_model)

    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[route_by_length],
)

In [ ]:
from langchain.messages import HumanMessage

# Short conversation → should use gpt-5-nano
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 2 + 2?")]}
)
print("Model used:", response["messages"][-1].response_metadata["model_name"])
print("Answer:", response["messages"][-1].content)

In [ ]:
from langchain.messages import AIMessage

# Long conversation (11 messages) → should switch to claude-sonnet-4-5
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Q1"), AIMessage(content="A1"),
        HumanMessage(content="Q2"), AIMessage(content="A2"),
        HumanMessage(content="Q3"), AIMessage(content="A3"),
        HumanMessage(content="Q4"), AIMessage(content="A4"),
        HumanMessage(content="Q5"), AIMessage(content="A5"),
        HumanMessage(content="What model are you?"),
    ]}
)
print("Model used:", response["messages"][-1].response_metadata["model_name"])

### 🛡️ Example 1B: Dynamic Tool Filtering

`wrap_model_call` can also **restrict which tools** the LLM sees based on runtime context. This is powerful for **role-based access control** — internal users get all tools, external users get a safe subset.

The key method: `request.override(tools=filtered_list)`

In [ ]:
from langchain.tools import tool
from dataclasses import dataclass


@tool
def web_search(query: str) -> str:
    """Search the web for public information."""
    return f"Web results for: {query}"


@tool
def internal_db_query(sql: str) -> str:
    """Query the internal database. Restricted to internal users."""
    return f"DB results for: {sql}"


@dataclass
class UserRole:
    user_role: str = "external"  # "internal" or "external"

In [ ]:
@wrap_model_call
def filter_tools_by_role(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """External users can only use web_search. Internal users get all tools."""
    user_role = request.runtime.context.user_role

    if user_role != "internal":
        # Strip internal tools — external users only see web_search
        safe_tools = [t for t in request.tools if t.name == "web_search"]
        request = request.override(tools=safe_tools)

    return handler(request)


agent_rbac = create_agent(
    model="gpt-5-nano",
    tools=[web_search, internal_db_query],
    middleware=[filter_tools_by_role],
    context_schema=UserRole,
)

In [ ]:
# External user tries to query the database — agent won't have that tool
response = agent_rbac.invoke(
    {"messages": [HumanMessage(content="How many users are in the database?")]},
    context=UserRole(user_role="external"),
)
print("External user:", response["messages"][-1].content)

---

## 🔀 Part 2: `wrap_tool_call`

### What it does

Same pattern as `wrap_model_call`, but wraps **tool execution** instead of the LLM call. You receive a `ToolCallRequest` and a `handler`.

### Use cases

| Use case | How |
|---|---|
| 🔒 Authorization | Block tool calls based on user role |
| 📊 Logging/metrics | Time how long each tool takes |
| 💾 Caching | Return cached results for repeated calls |
| 🔁 Retry logic | Retry failed tool calls with backoff |
| 🛡️ Input validation | Sanitize tool arguments before execution |

### The pattern

```python
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest

@wrap_tool_call
def my_tool_wrapper(
    request: ToolCallRequest,
    handler: Callable
):
    # request.tool_name — which tool is being called
    # request.tool_args — the arguments
    # request.runtime   — access to context, state
    
    # Optionally modify, block, or cache
    result = handler(request)
    # Optionally transform the result
    return result
```

### Example: Logging tool calls

In [ ]:
import time
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest


@wrap_tool_call
def log_tool_calls(request: ToolCallRequest, handler: Callable):
    """Log every tool call with timing."""
    print(f"  🔨 Calling tool: {request.tool_name}")
    print(f"     Args: {request.tool_args}")

    start = time.time()
    result = handler(request)
    elapsed = time.time() - start

    print(f"  ✅ Tool {request.tool_name} completed in {elapsed:.2f}s")
    return result


agent_logged = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    middleware=[log_tool_calls],
)

In [ ]:
response = agent_logged.invoke(
    {"messages": [HumanMessage(content="Search for LangChain middleware docs")]}
)
print("\nFinal answer:", response["messages"][-1].content)

---

## ⚙️ Part 3: `dynamic_prompt`

### What it does

Generates the **system prompt dynamically** based on runtime state — the conversation so far, the user's context, the current time, etc. Instead of a static `system_prompt="..."` string, you write a function that returns the prompt.

### Why not just use a static prompt?

| Static prompt | Dynamic prompt |
|---|---|
| Same for every user | Personalized per user |
| Same every turn | Can change mid-conversation |
| Can't read context | Reads `runtime.context` |
| Can't read state | Reads `request.messages`, `request.runtime.state` |

### Real-world examples

- 🌍 **Language routing** — respond in the user's preferred language
- 👤 **Role-based behavior** — admin vs regular user gets different instructions
- 🕐 **Time-aware** — "It's after business hours, suggest the FAQ instead"
- 📊 **State-aware** — "The user has already provided X, don't ask again"

### The pattern

```python
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def my_prompt(request: ModelRequest) -> str:
    # Read context, state, messages — anything you need
    user_lang = request.runtime.context.user_language
    return f"You are a helpful assistant. Respond in {user_lang}."
```

> 💡 `@dynamic_prompt` **replaces** the `system_prompt` parameter in `create_agent()`. Don't set both — the dynamic one wins.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class LanguageContext:
    user_language: str = "English"


@dynamic_prompt
def language_aware_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user's preferred language."""
    lang = request.runtime.context.user_language
    base = "You are a helpful assistant."

    if lang != "English":
        return f"{base} You MUST respond only in {lang}."
    return base


agent_lang = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[language_aware_prompt],
    # NOTE: no system_prompt= here — dynamic_prompt replaces it
)

In [ ]:
# English context → normal English response
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="English"),
)
print("English:", response["messages"][-1].content)

In [ ]:
# Spanish context → response in Spanish
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish"),
)
print("Spanish:", response["messages"][-1].content)

In [ ]:
# Japanese context → response in Japanese
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Japanese"),
)
print("Japanese:", response["messages"][-1].content)

---

## 🗺️ Complete Middleware Cheat Sheet

### Decorator-based middleware

| Decorator | Family | Intercepts | Key method | Use case |
|---|---|---|---|---|
| `@before_agent` | Node | State before LLM | Return `dict` | Trim/summarize messages |
| `@after_agent` | Node | State after LLM | Return `dict` | Log/validate responses |
| `@before_model` | Node | State before model | Return `dict` | Similar to before_agent |
| `@after_model` | Node | State after model | Return `dict` | Similar to after_agent |
| `@wrap_model_call` | Wrap | LLM invocation | `request.override()` | Swap models, filter tools |
| `@wrap_tool_call` | Wrap | Tool execution | `handler(request)` | Log, cache, authorize |
| `@dynamic_prompt` | Config | Prompt generation | Return `str` | Per-user/per-turn prompts |

### Built-in middleware classes

| Class | What it does |
|---|---|
| `SummarizationMiddleware` | Auto-summarize old messages |
| `PIIMiddleware` | Detect/redact personally identifiable information |
| `HumanInTheLoopMiddleware` | Pause for human approval before tool execution |
| `ModelCallLimitMiddleware` | Cap the number of LLM calls per session |
| `ModelFallbackMiddleware` | Fall back to another model on error |
| `ModelRetryMiddleware` | Retry failed LLM calls with backoff |
| `ToolCallLimitMiddleware` | Cap the number of tool calls per session |
| `ToolRetryMiddleware` | Retry failed tool calls |
| `ShellToolMiddleware` | Sandbox shell command execution |
| `ContextEditingMiddleware` | Edit/clear tool usage from context |
| `LLMToolSelectorMiddleware` | Use an LLM to select which tools to expose |
| `LLMToolEmulator` | Emulate tool behavior with an LLM |
| `TodoListMiddleware` | Track tasks in a todo list |
| `FilesystemFileSearchMiddleware` | Search files on the filesystem |

---

## 💡 Key Takeaways

1. **`wrap_model_call`** = wrap the LLM call. You see request AND response. Best for model routing, tool filtering, retries.

2. **`wrap_tool_call`** = wrap tool execution. Same pattern. Best for logging, caching, authorization.

3. **`dynamic_prompt`** = generate the system prompt at runtime. Best for personalization, language routing, role-based behavior.

4. **All three** can read `request.runtime.context` (read-only user data) and `request.runtime.state` (mutable agent state).

5. **Combine them freely** — you can use node-style, wrap-style, and config-style middleware together in the same `middleware=[]` list. They compose naturally.